# Ingestion and Labeling Notebook

TODO: Add notebook introduction

## Environment Setup

TODO: Briefly outline environment setup

In [7]:
# Import libraries

# Standard Library
import sys
from pathlib import Path
import pandas as pd
from collections import Counter

# Repository Root Setup
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

# Custom Modules
from src.config import load_settings
from src.spark.session import get_spark
from src.spark.ingestion.spark.transforms.recipes_spark import clean_recipes
from src.spark.ingestion.spark.transforms.interactions_spark import clean_interactions
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_reviews
from src.spark.labeling.zero_shot import (
    ZeroShotConfig, 
    prepare_zero_shot_input_from_gold, 
    run_zero_shot_incremental_with_checkpoints,
    attach_zero_shot_labels_to_gold,
    run_final_label_report
)

# Initialize Settings and Spark
s = load_settings(prefer_latest_run=True)
spark = get_spark(app_name="01_ingestion_and_labeling", debug=True)

print(f"Environment Ready: {s.env}")
print(f"Processed Dir: {s.processed_dir}")

# Spark
import pyspark.sql.functions as F

Environment Ready: local
Processed Dir: /home/iauger/projects/dsci632-project/data/processed


## Ingestion Logic

TODO: Describe process of downloading RAW and processing to Bronze and then to Silver

In [8]:
# Read Raw CSVs
recipes_raw = spark.read.option("header", "true").option("multiline", "true").csv(s.raw_recipes_path)
interactions_raw = spark.read.option("header", "true").option("multiline", "true").csv(s.raw_interactions_path)

# Clean and Move to Silver
recipes_s = clean_recipes(recipes_raw)
interactions_s = clean_interactions(interactions_raw)

# Create the Base 'Gold' Reviews (includes the review_key hash)
gold_reviews = build_gold_reviews(interactions_s)

# Save to Disk
gold_reviews.write.mode("overwrite").parquet(s.gold_reviews_path)
print(f"Silver/Gold Ingestion Complete: {gold_reviews.count():,} reviews.")

Silver/Gold Ingestion Complete: 1,070,664 reviews.


In [9]:
# Block 2.1: Pre-Labeling Data Quality Check
print("--- Silver Review Quality Metrics ---")
# 1. Check for Nulls/Empties that break transformers
gold_reviews = spark.read.parquet(s.gold_reviews_path)
gold_reviews.select(
    F.sum(F.col("review_clean").isNull().cast("int")).alias("null_reviews"),
    F.sum((F.length(F.trim(F.col("review_clean"))) == 0).cast("int")).alias("empty_strings")
).show()

# 2. Token Length Distribution
# This confirms if our min_tokens=15 rule is too restrictive
review_stats = gold_reviews.withColumn("tokens", F.size(F.split(F.col("review_clean"), r"\s+")))
print("--- Review Token Percentiles ---")
review_stats.selectExpr("percentile_approx(tokens, array(0.1, 0.25, 0.5, 0.75, 0.9)) as p").show()

--- Silver Review Quality Metrics ---


+------------+-------------+
|null_reviews|empty_strings|
+------------+-------------+
|           0|          215|
+------------+-------------+

--- Review Token Percentiles ---


+--------------------+
|                   p|
+--------------------+
|[16, 28, 44, 67, 96]|
+--------------------+



## Zero-Shot Labeling

TODO: Outline the why here, the approach, and how labels are being stored/updated

In [10]:
from pyspark.sql import functions as F

# Define keyword buckets for low occurence tags
sinkhole_keywords = {
    "too_spicy": ["spicy", "hot", "cayenne", "chili", "burn", "heat", "kick"],
    "too_salty": ["salty", "salt", "sodium", "brine"],
    "too_acidic": ["acidic", "vinegar", "lemon", "lime", "tart", "sour", "tangy"],
    "too_sweet": ["sweet", "sugar", "syrup", "cloying", "honey"],
    "mushy_soggy": ["mushy", "soggy", "wet", "limp", "texture"],
    "bland_lacks_flavor": ["bland", "tasteless", "flavorless", "boring", "plain"]
}

# Combine all keywords into a single regex for efficient filtering
all_keywords = [kw for sublist in sinkhole_keywords.values() for kw in sublist]
regex_pattern = "|".join([rf"\b{kw}\b" for kw in all_keywords])

# Filter the Silver Corpus
gold_reviews = spark.read.parquet(s.gold_reviews_path)

target_candidates = gold_reviews.filter(
    F.col("review_clean").rlike(regex_pattern)
)

# Sample 60k rows 
labeling_sample_60k = target_candidates.sample(False, 60000 / target_candidates.count(), seed=42).limit(60000)
sample_out_path = Path(s.gold_dir) / "labeling_sample_60k.parquet"
labeling_sample_60k.write.mode("overwrite").parquet(str(sample_out_path))

print(f"Prepared {labeling_sample_60k.count()} keyword-targeted rows for labeling.")

Prepared 60000 keyword-targeted rows for labeling.


In [11]:
# Set to true to run on the keyword-targeted sample instead of the full gold set
ALTERNATE_SAMPLE = True

# Configure Zero-Shot Labeling
cfg = ZeroShotConfig(
    taxonomy_version="v1",
    min_tokens=15,
    sample_n=60000,   
    batch_size=4,     
    sample_seed=42
)

if ALTERNATE_SAMPLE:
    print("Running on the alternate keyword-targeted sample for zero-shot labeling.")
    model_path = str(sample_out_path)
else:
    print("Running on the full gold set for zero-shot labeling.")
    model_path = str(s.gold_reviews_path)
    
# Prepare/Check Input
inp_path = prepare_zero_shot_input_from_gold(spark, model_path, cfg, s.processed_dir)

# Run Incremental 
print("Starting incremental zero-shot run...")
out_path = run_zero_shot_incremental_with_checkpoints(inp_path, cfg, s.processed_dir)

Running on the alternate keyword-targeted sample for zero-shot labeling.


Starting incremental zero-shot run...


Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Zero-shot batches:   0%|          | 0/125 [00:00<?, ?batch/s]

...

Zero-shot batches:   0%|          | 0/125 [00:00<?, ?batch/s]

Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Zero-shot batches:   0%|          | 0/125 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [12]:
# Sync to the final Gold Labeled file
labeled_path = attach_zero_shot_labels_to_gold(
    spark, 
    model_path, 
    s.processed_dir, 
    cfg, 
    out_path=f"{s.processed_dir}/labeling/zero_shot/labeled_gold_reviews_v6.parquet"
)

run_final_label_report(labeled_path)

--- Final Labeling Report: labeled_gold_reviews_v6.parquet ---
Total Labeled Samples: 12,810
Avg Labels/Review: 3.23

Top Tags (% of dataset):
delicious_tasty............... 9,472 (73.94%)
would_make_again.............. 9,176 (71.63%)
family_hit.................... 7,750 (60.50%)
substitution_modification..... 6,957 (54.31%)
easy_quick.................... 3,454 (26.96%)
would_not_make_again.......... 1,142 (8.91%)
ingredient_issue.............. 1,063 (8.30%)
bland_lacks_flavor............ 820 (6.40%)
time_consuming_complex........ 447 (3.49%)
moist_tender.................. 337 (2.63%)

Reviews with 0 labels: 825 (6.44%)


In [15]:
from pyspark.sql import functions as F

# 1. Load both sets
v5_path = f"{s.processed_dir}/labeling/zero_shot/labeled_gold_reviews_v5.parquet"
v6_path = f"{s.processed_dir}/labeling/zero_shot/labeled_gold_reviews_v6.parquet"

df_v5 = spark.read.parquet(v5_path)
df_v6 = spark.read.parquet(v6_path)

# 2. Combine and Deduplicate (just in case some 10k rows overlapped)
# 'review_key' is your primary unique identifier
combined_labeled_df = df_v5.unionByName(df_v6).dropDuplicates(["review_key"])

# 3. Overwrite v6 to establish the new 'Master' labeled set
# Repartitioning to 20 helps keep the write stable and efficient for future loading
combined_labeled_df.repartition(20).write.mode("overwrite").parquet(v6_path)

run_final_label_report(v6_path)

--- Final Labeling Report: labeled_gold_reviews_v6.parquet ---
Total Labeled Samples: 49,446
Avg Labels/Review: 3.22

Top Tags (% of dataset):
delicious_tasty............... 38,336 (77.53%)
would_make_again.............. 37,197 (75.23%)
family_hit.................... 32,550 (65.83%)
substitution_modification..... 22,675 (45.86%)
easy_quick.................... 14,734 (29.80%)
would_not_make_again.......... 3,521 (7.12%)
ingredient_issue.............. 3,363 (6.80%)
bland_lacks_flavor............ 2,123 (4.29%)
time_consuming_complex........ 1,511 (3.06%)
moist_tender.................. 1,382 (2.79%)

Reviews with 0 labels: 2986 (6.04%)


## Conclusion

TODO: Review the full ingestion to labeling pipeline and how this sets up the upcoming stages. 